# Stage B1 — Architecture Search

Dual-fuel PINN pipeline | Sandrine Schueller Mafra | PPGEM – UFPR
Supports dissertation Section 3.2.1.2 (Table 6), feedback item asking
to connect Table 6 to a design-of-experiments framing and to justify
the winning architecture in terms of Occam's razor.

Self-contained: reloads and re-splits `masters_data.xlsx` exactly as A3
does (same code, repeated here) rather than depending on A3 having
been run first.

**Framework note:** this stage trains actual neural networks, so it
needs `tensorflow`/`keras` — `polars`/`plotly` alone can't do that.
Everything else (data handling, plotting) stays in polars/plotly as in
A1-A3.

**No physics here yet.** The 12 candidates below are plain MLPs —
`tanh` hidden layers, linear output, MSE loss, nothing else. That's
deliberate: Table 6 / Sec. 3.2.1 is the *baseline* architecture search.
The physics constraints (Table 7, Eq. 3.4–3.15) and the composite
physics-informed loss (Eq. 3.16) are Sec. 3.2.2 onward — **Stages C1
(constraints) and C2 (loss assembly)**, not this one. What B1 selects
is the backbone *shape* (layers/neurons) that both B2 (baseline,
trained exactly as here) and C3/C4 (same shape, physics loss added on
top) will reuse — the architecture search and the physics are
deliberately separated so "does the shape help" and "does the physics
help" can be answered independently instead of tangled together.

**Input:** `data/masters_data.xlsx`
**What this notebook does:**
1. Rebuilds the OFAT-blocked, extremity-stratified split and train-only
   normalization from A3.
2. Defines the 12 candidate architectures from **Table 6**, plus **4
   proposed additions** (Section 2b — not in the dissertation text,
   reasoning given there), and checks every *reported* Table 6
   parameter count against what Keras actually computes for that shape.
3. Runs 5-fold cross-validation × 3 seeds (15 fits per architecture,
   **240 fits total** across all 16) over the 34 train+val points —
   **test stays held out**, untouched until Phase D.
4. Picks a winner not by lowest error alone, but by the *simplest*
   architecture that isn't statistically distinguishable from the best
   one (paired Wilcoxon test) — the Occam's-razor criterion the text
   already commits to in words.

**Output:** a results table (mean ± std MSE/R² per architecture) and
one selected architecture, used by B2 and, later, C3/C4.

**Runtime:** 240 small model fits on 34 points each — expect several
minutes, not seconds. Reduce `SEEDS` to `[0]` for a quick smoke test
before running the full search.


## Setup

In [ ]:
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from sklearn.model_selection import KFold
from scipy.stats import wilcoxon
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from time import sleep
from tqdm import tqdm

print("polars    ", pl.__version__)
import plotly
print("plotly    ", plotly.__version__)
print("tensorflow", tf.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "masters_data.xlsx"
OUT_DIR = PROJECT_ROOT / "outputs"
SEED = 42
RAW_PATH


## 1. Load, split, normalize (same logic as A3)

Repeated here rather than imported so this notebook runs standalone.
See A3 for the extremity-based split rationale.

In [ ]:
COLUMN_MAP = {
    "SOI [o.CA]": "SOI", "Lambda [-]": "lambda", "Sub. Rate [%]": "sub_rate",
    "Prail [bar]": "P_rail", "HC [g/kW.h]": "HC", "NOX [ppm]": "NOx",
    "CO2 [%]": "CO2", "SO_H [FSN]": "PM", "ETA [%]": "eta",
}
INPUT_COLS = ["SOI", "lambda", "sub_rate", "P_rail"]
OUTPUT_COLS = ["HC", "NOx", "CO2", "PM", "eta"]
ALL_COLS = INPUT_COLS + OUTPUT_COLS

df = pl.read_excel(RAW_PATH).rename(COLUMN_MAP).select(ALL_COLS)
n = df.shape[0]

# OFAT block (median-deviation + smoothing)
medians = {c: df[c].median() for c in INPUT_COLS}
ranges = {c: (df[c].max() - df[c].min()) for c in INPUT_COLS}
deviation = np.column_stack([np.abs(df[c].to_numpy() - medians[c]) / ranges[c] for c in INPUT_COLS])
raw_block = np.array(INPUT_COLS)[deviation.argmax(axis=1)]

def smooth_isolated_labels(labels, passes=2):
    out = list(labels)
    for _ in range(passes):
        changed = False
        for i in range(1, len(out) - 1):
            if out[i] != out[i - 1] and out[i - 1] == out[i + 1]:
                out[i] = out[i - 1]
                changed = True
        if not changed:
            break
    return np.array(out)

ofat_block = smooth_isolated_labels(raw_block)

# extremity-based split
extremity = np.zeros(n)
for b in np.unique(ofat_block):
    idx = np.where(ofat_block == b)[0]
    vals = df[b].to_numpy()[idx]
    order = np.argsort(vals)
    m = len(idx)
    pos = np.array([0.5]) if m == 1 else np.empty(m)
    if m > 1:
        ranks = np.empty(m)
        ranks[order] = np.arange(m)
        pos = ranks / (m - 1)
    extremity[idx] = np.abs(pos - 0.5) * 2

rng = np.random.default_rng(SEED)
jitter = rng.uniform(-1e-9, 1e-9, size=n)
order = np.argsort(-(extremity + jitter))
split = np.array(["train"] * n)
split[order[:6]] = "test"
split[order[6:12]] = "val"
df = df.with_columns(pl.Series("split", split))

# min-max normalization fit on train only
train = df.filter(pl.col("split") == "train")
train_min = {c: train[c].min() for c in ALL_COLS}
train_max = {c: train[c].max() for c in ALL_COLS}
df = df.with_columns([
    ((pl.col(c) - train_min[c]) / (train_max[c] - train_min[c])).alias(f"{c}_norm")
    for c in ALL_COLS
])

print(dict(zip(*np.unique(split, return_counts=True))))
df.head(3)


## 2a. Candidate architectures from Table 6

Architecture is defined by hidden-layer widths alone; parameter count
is *computed*, not copied from the table, and compared against what
Table 6 states. Mismatches are flagged rather than assumed away — this
follows the same principle as A1's Table 4/5 check.

In [ ]:
ARCHITECTURES_TABLE6 = {
    "A1": (16,), "A2": (32,), "A3": (64,),
    "A4": (16, 16), "A5": (32, 32), "A6": (64, 64),
    "A7": (16, 8), "A8": (32, 16),
    "A9": (16, 16, 16), "A10": (32, 32, 32), "A11": (64, 32, 16),
    "A12": (16, 16, 16, 16),
}
TABLE6_PARAMS = {  # as printed in the dissertation text, Sec. 3.2.1.2
    "A1": 165, "A2": 325, "A3": 645, "A4": 421, "A5": 1317, "A6": 4933,
    "A7": 277, "A8": 725, "A9": 661, "A10": 2469, "A11": 3317, "A12": 949,
}
N_IN, N_OUT = len(INPUT_COLS), len(OUTPUT_COLS)

def count_params(hidden, in_dim=N_IN, out_dim=N_OUT):
    total, prev = 0, in_dim
    for h in hidden:
        total += prev * h + h
        prev = h
    total += prev * out_dim + out_dim
    return total

rows = []
for aid, hidden in ARCHITECTURES_TABLE6.items():
    computed = count_params(hidden)
    stated = TABLE6_PARAMS[aid]
    rows.append({
        "architecture": aid, "hidden_layers": len(hidden), "shape": str(hidden),
        "computed_params": computed, "table6_params": stated,
        "status": "match" if computed == stated else f"MISMATCH ({computed - stated:+d})",
    })
arch_table = pl.DataFrame(rows)
arch_table


**Equation implemented above** — parameters of a fully-connected
stack with hidden widths $n_1, \dots, n_L$, input width $n_0=4$ and
output width $n_{L+1}=5$ (bias term per layer):

$$
N_{\text{params}} = \sum_{l=1}^{L+1} \left( n_{l-1} \cdot n_l + n_l \right)
$$

Shown against what Table 6 states, per architecture:

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(x=arch_table["architecture"], y=arch_table["computed_params"],
                      name="computed (this notebook)", marker_color="#185FA5"))
fig.add_trace(go.Bar(x=arch_table["architecture"], y=arch_table["table6_params"],
                      name="Table 6 (text)", marker_color="#993C1D"))
fig.update_layout(barmode="group", title="Parameter count: computed vs. Table 6",
                   yaxis_title="parameters", width=800, height=420)
fig.show()


## 2b. Proposed additions — not in Table 6

Table 6's 12 candidates already span 165 to 4933 parameters (6x to
178x the ~27 points in a single training fold), across 1-4 hidden
layers and widths 8-64. Four gaps are worth closing before trusting
that grid, each for a specific reason — not a blind "add more":

| id | shape | why |
|---|---|---|
| **A13_linear** | none (linear map) | Floor check. Every Table 6 candidate is already 6x+ over-parameterized for a 27-point training fold; worth confirming a plain linear model isn't *already* competitive before trusting any nonlinear one. |
| **A14_tiny4** | 1×4 | Table 6's smallest is 16 units (A1). Nothing tests the gap between that and a linear map — is there a smooth transition or a cliff? |
| **A15_tiny8** | 1×8 | Same purpose, one step up from A14. |
| **A16_wide128** | 1×128 | Ceiling check for a *single* layer: Table 6 stops at 64 (A3). Completes the width sweep 4→8→16→32→64→128 instead of assuming 64 was already the top of the useful range. |

**Deliberately not proposing:** deeper (5+ layer) or larger multi-layer
combinations. Table 6 already reaches 178x over-parameterized at A6;
going bigger multiplies runtime for a question the existing grid's own
results (Section 6 below) already lean against — more capacity hasn't
clearly helped so far, and 34 points won't support much more before
these become memorization exercises rather than fits.

In [ ]:
ARCHITECTURES_NEW = {
    "A13_linear": (),
    "A14_tiny4": (4,),
    "A15_tiny8": (8,),
    "A16_wide128": (128,),
}
ARCHITECTURES = {**ARCHITECTURES_TABLE6, **ARCHITECTURES_NEW}

new_rows = [{"architecture": aid, "hidden_layers": len(hidden), "shape": str(hidden) if hidden else "(linear)",
             "computed_params": count_params(hidden)}
            for aid, hidden in ARCHITECTURES_NEW.items()]
pl.DataFrame(new_rows)


## 3. Model builder & CV protocol

All 5 outputs predicted jointly (multi-output regression) from the 4
normalized inputs; `tanh` activation throughout, matching the text's
justification in Sec. 3.2.1.3 (bounded, smooth derivatives — relevant
again once Phase C needs gradients through this same architecture).
Test rows are excluded from the CV pool entirely.

**Equation implemented for every hidden layer $l$:**

$$
h^{(l)} = \tanh\!\left( W^{(l)} h^{(l-1)} + b^{(l)} \right), \qquad
\hat{y} = W^{(L+1)} h^{(L)} + b^{(L+1)}
$$

(linear, not tanh, on the output layer — this is a regression head,
not a classifier).

In [ ]:
def build_model(hidden_units, seed, input_dim=N_IN, output_dim=N_OUT):
    tf.random.set_seed(seed)
    model = keras.Sequential([keras.Input(shape=(input_dim,))])
    for units in hidden_units:
        model.add(layers.Dense(units, activation="tanh"))
    model.add(layers.Dense(output_dim, activation="linear"))
    model.compile(optimizer="adam", loss="mse")
    return model

cv_pool = df.filter(pl.col("split") != "test")
X = cv_pool.select([f"{c}_norm" for c in INPUT_COLS]).to_numpy()
Y = cv_pool.select([f"{c}_norm" for c in OUTPUT_COLS]).to_numpy()
print("CV pool:", X.shape[0], "points (test held out separately)")

SEEDS = [0, 1, 2]
N_FOLDS = 5
EPOCHS = 300
PATIENCE = 30
BATCH_SIZE = 8


**What the 5 folds actually look like**, for seed 0 (every architecture reuses these same splits, which is what makes the paired test in Section 8 valid):

In [ ]:
fold_map = np.full((N_FOLDS, X.shape[0]), "train", dtype=object)
kf0 = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEEDS[0])
for fold_i, (tr_idx, va_idx) in enumerate(kf0.split(X)):
    fold_map[fold_i, va_idx] = "validation"

z = (fold_map == "validation").astype(int)
fig = go.Figure(go.Heatmap(
    z=z, x=[f"pt {i}" for i in range(X.shape[0])], y=[f"fold {i}" for i in range(N_FOLDS)],
    colorscale=[[0, "#E6F1FB"], [1, "#185FA5"]], showscale=False,
))
fig.update_layout(title="Train (light) vs. validation (dark) per fold, seed 0",
                   width=850, height=280, xaxis_showticklabels=False)
fig.show()


## 4. Run the search — 16 architectures × 3 seeds × 5 folds = 240 fits

This is the slow cell — noticeably longer now with `A16_wide128` in
the mix (128-unit layer, still fast per-fit, but 20 of the 240 fits
are its). `keras.backend.clear_session()` runs every iteration to stop
TensorFlow's graph/memory bookkeeping from growing across 240
freshly-built models.

In [ ]:
results = []
for arch_id, hidden in tqdm(ARCHITECTURES.items()):
    for seed in tqdm(SEEDS):
        kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for fold_i, (tr_idx, va_idx) in tqdm(enumerate(kf.split(X))):
            keras.backend.clear_session()
            model = build_model(hidden, seed=seed * 100 + fold_i)
            es = keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=PATIENCE, restore_best_weights=True
            )
            model.fit(
                X[tr_idx], Y[tr_idx],
                validation_data=(X[va_idx], Y[va_idx]),
                epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0, callbacks=[es],
            )
            pred = model.predict(X[va_idx], verbose=0)
            mse = float(np.mean((pred - Y[va_idx]) ** 2))
            ss_res = float(np.sum((Y[va_idx] - pred) ** 2))
            ss_tot = float(np.sum((Y[va_idx] - Y[va_idx].mean(axis=0)) ** 2))
            r2 = 1 - ss_res / ss_tot
            results.append(dict(architecture=arch_id, seed=seed, fold=fold_i, mse=mse, r2=r2))
    print(f"{arch_id} done")

cv_results = pl.DataFrame(results)
OUT_DIR.mkdir(parents=True, exist_ok=True)
cv_results.write_csv(OUT_DIR / "B1_cv_results_raw.csv")  # checkpoint, this cell is slow
cv_results.head()


## 5. Aggregate results per architecture

**Equations implemented per fold** (both computed on the normalized
outputs, over the held-out fold's $N$ points and all 5 outputs):

$$
\text{MSE} = \frac{1}{5N}\sum_{i=1}^{N}\sum_{k=1}^{5}\left(y_{i,k}-\hat y_{i,k}\right)^2,
\qquad
R^2 = 1 - \frac{\sum_{i,k}\left(y_{i,k}-\hat y_{i,k}\right)^2}{\sum_{i,k}\left(y_{i,k}-\bar y_{k}\right)^2}
$$

then averaged (mean ± std) over the 15 seed×fold fits per architecture.

In [ ]:
summary_rows = []
for aid in ARCHITECTURES:
    sub = cv_results.filter(pl.col("architecture") == aid)
    mse_vals = sub["mse"].to_numpy()
    r2_vals = sub["r2"].to_numpy()
    summary_rows.append({
        "architecture": aid,
        "n_params": count_params(ARCHITECTURES[aid]),
        "mean_mse": mse_vals.mean(), "std_mse": mse_vals.std(ddof=1),
        "mean_r2": r2_vals.mean(), "std_r2": r2_vals.std(ddof=1),
    })
summary = pl.DataFrame(summary_rows).sort("mean_mse")
summary


## 6. Performance by architecture

In [ ]:
fig = go.Figure()
order_ids = summary["architecture"].to_list()
for aid in order_ids:
    vals = cv_results.filter(pl.col("architecture") == aid)["mse"].to_numpy()
    fig.add_trace(go.Box(y=vals, name=aid, marker_color="#185FA5", boxpoints="all",
                          jitter=0.4, pointpos=0, showlegend=False))
fig.update_layout(title="Validation MSE across 15 CV fits (3 seeds x 5 folds), by architecture",
                   yaxis_title="MSE (normalized outputs)", xaxis_title="architecture",
                   width=850, height=450)
fig.show()


## 7. Parameters vs. performance — the Occam's-razor view

If a small architecture sits at roughly the same height as a much
larger one, added complexity bought nothing. Diamonds are the four
proposed additions from Section 2b, circles are Table 6's original 12.

In [ ]:
is_new = [a in ARCHITECTURES_NEW for a in summary["architecture"].to_list()]
symbols = ["diamond" if flag else "circle" for flag in is_new]
colors = ["#854F0B" if flag else "#534AB7" for flag in is_new]

fig = go.Figure(go.Scatter(
    x=summary["n_params"], y=summary["mean_mse"],
    error_y=dict(type="data", array=summary["std_mse"], visible=True),
    mode="markers+text", text=summary["architecture"], textposition="top center",
    marker=dict(size=11, color=colors, symbol=symbols),
))
fig.update_layout(title="Mean CV MSE vs. parameter count (diamonds = proposed, error bars = 1 std)",
                   xaxis_title="parameters", yaxis_title="mean MSE", width=800, height=480)
fig.show()


## 8. Statistical comparison & final selection

The naive choice is whichever architecture has the lowest mean MSE.
Instead: take every architecture with *fewer* parameters than the
naive winner, run a paired Wilcoxon signed-rank test against it
(paired on matching seed+fold, since every architecture sees the same
15 CV splits), and if the difference isn't significant (p > 0.05),
that simpler architecture is a legitimate substitute. Among the naive
winner and all such substitutes, keep the one with the fewest
parameters.

In [ ]:
naive_winner = summary["architecture"][0]
winner_params = summary["n_params"][0]
winner_vals = (cv_results.filter(pl.col("architecture") == naive_winner)
               .sort(["seed", "fold"])["mse"].to_numpy())

candidates = [(naive_winner, winner_params)]
comparisons = []  # for the chart in the next cell
print(f"Naive winner (lowest mean MSE): {naive_winner}  (params={winner_params})\n")
print("Paired Wilcoxon vs. naive winner, architectures with fewer parameters:")
for aid in summary["architecture"].to_list():
    if aid == naive_winner:
        continue
    p_count = count_params(ARCHITECTURES[aid])
    if p_count >= winner_params:
        continue
    vals = cv_results.filter(pl.col("architecture") == aid).sort(["seed", "fold"])["mse"].to_numpy()
    stat, p = wilcoxon(winner_vals, vals)
    verdict = "not significant -> valid substitute" if p > 0.05 else "significant -> keep winner"
    print(f"  {aid:4s} (params={p_count:4d})  p={p:.4f}   {verdict}")
    comparisons.append({"architecture": aid, "p_value": p, "significant": p <= 0.05})
    if p > 0.05:
        candidates.append((aid, p_count))

selected = min(candidates, key=lambda t: t[1])
print(f"\nSELECTED ARCHITECTURE: {selected[0]}  ({selected[1]} parameters)")
print("Simplest architecture not statistically distinguishable from the naive winner.")


**Same comparison, visually** — bars crossing the dashed line are statistically indistinguishable from the naive winner, i.e. valid simpler substitutes:

In [ ]:
comp_df = pl.DataFrame(comparisons).sort("p_value", descending=True)
bar_colors = ["#3B6D11" if not sig else "#993C1D" for sig in comp_df["significant"].to_list()]

fig = go.Figure(go.Bar(x=comp_df["architecture"], y=comp_df["p_value"], marker_color=bar_colors))
fig.add_hline(y=0.05, line_dash="dash", line_color="#5A5A55", annotation_text="p = 0.05")
fig.update_layout(
    title=f"Paired Wilcoxon p-value vs. {naive_winner} (green = valid simpler substitute)",
    yaxis_title="p-value", xaxis_title="architecture (fewer params than the naive winner)",
    width=750, height=420,
)
fig.show()


## Optional — persist final outputs

Saves the full comparison table **and** the selected architecture's
identity — B2 reads the latter to know which shape to train, so this
cell isn't really optional if you intend to run B2 next.

In [ ]:
import json

summary.write_csv(OUT_DIR / "B1_architecture_summary.csv")

selection_record = {
    "architecture_id": selected[0],
    "hidden_units": list(ARCHITECTURES[selected[0]]),
    "n_params": selected[1],
    "is_proposed_addition": selected[0] in ARCHITECTURES_NEW,
}
with open(OUT_DIR / "B1_selected_architecture.json", "w") as f:
    json.dump(selection_record, f, indent=2)

print(f"Saved to {OUT_DIR}")
print(f"Selected architecture: {selected[0]} -> hidden layers {ARCHITECTURES[selected[0]]}")


## Next

**B2** trains the selected architecture once (no physics terms) on the
28-point train split, validated on the 6-point val split, as the
baseline that Phase D compares the PINN against — it reads
`B1_selected_architecture.json` from this notebook's save cell, so run
that cell before starting B2. The same architecture shape also becomes
the starting point for **C3**'s hyperparameter search (Table 8), which
adds the physics-loss weighting on top.
